# Pipeline

> End-to-end pipeline for processing IOM evaluation reports


The `Report` class orchestrates the full evaluation pipeline:

1. **Download** - Fetch PDFs from IOM evaluation repository
2. **OCR** - Convert PDFs to markdown with heading hierarchy
3. **Curate** - Manually review headings and select relevant sections (via `curator` app)
4. **Map** - Identify which standardized framework themes are relevant to each report, including the [Strategic Results Framework (SRF)](https://srf.iom.int) (Enablers, Cross-cutting Priorities, and Outputs) and [Global Compact for Migration](https://www.un.org/en/development/desa/population/migration/generalassembly/docs/globalcompact/A_RES_73_195.pdf) objectives

In [ ]:
#| default_exp pipeline

In [ ]:
#| export
from fastcore.all import *
from pathlib import Path
import logging
from iomeval.core import n_tokens, load_prompt
from iomeval.readers import load_evals, find_eval, Evaluation, eval_url
from iomeval.downloaders import download_eval
from iomeval.extract import extract_sections
from iomeval.themes import load_enbs, load_ccps, load_gcms, load_srf_outs, load_gcms_lut, fmt_enbs, fmt_ccps, fmt_srf_outs, get_srf_outs
from iomeval.mapper import mk_system_blocks, map_themes, sort_by_relevance, get_top_ids, parse_res
from mistocr.core import read_pgs
from mistocr.pipeline import pdf_to_md
from datetime import datetime
import json
import httpx
import shutil
import logging

In [ ]:
from cachy import enable_cachy
enable_cachy()

In [ ]:
#| export
logger = logging.getLogger(__name__)
logging.basicConfig(level=logging.WARNING, format='%(name)s - %(levelname)s - %(message)s')
logger.setLevel(logging.DEBUG)

## Report class

The `Report` class wraps an `Evaluation` and provides methods for the pipeline stages: download → ocr → curate → map

In [ ]:
#| export
#| exports
class Report:
    "An evaluation report with full pipeline support"
    def __init__(
        self,
        ev:Evaluation,                   # The evaluation metadata object
        pdf_url:str=None,                # Optional direct URL to PDF
        results_path:str='data/results'  # Path to save/load results
        ):
        store_attr()
        self.id = ev.id
        self.pdf_path,self.md_path = None,None
        self.mappings,self.selected_headings = {},[]
        self.curation_status = 'pending'
        self._load_existing()

In [ ]:
#| export
@patch
def _load_existing(self:Report):
    "Load state from saved JSON if it exists"
    p = Path(self.results_path)/f'{self.id}.json'
    if not p.exists(): return
    data = json.loads(p.read_text())
    self.mappings = data.get('mappings', {})
    self.curation_status = data.get('curation_status', 'pending')
    self.selected_headings = data.get('selected_headings', [])


In [ ]:
#| export
@patch(cls_method=True)
def from_url(
    cls:Report,
    url:str,                         # URL of the evaluation PDF
    evals:list,                      # List of `Evaluation` objects to search
    results_path:str='data/results'  # Path to save/load results
    ):                               # Report initialized from URL
    "Create a Report by finding an evaluation matching the given URL"
    return cls(find_eval(evals, url, by='url'), pdf_url=url, results_path=results_path)

In [ ]:
#| export
@patch(cls_method=True)
def from_title(
    cls:Report,
    title:str,                      # Title to search for
    evals:list,                     # List of `Evaluation` objects to search
    results_path:str='data/results' # Path to save/load results
    ):                               # Report initialized from title
    "Create a Report by finding an evaluation matching the given title"
    ev = find_eval(evals, title, by='title')
    return cls(ev, pdf_url=eval_url(ev), results_path=results_path)

In [ ]:
#| export
@patch(cls_method=True)
def from_id(
    cls:Report,
    id:str,                         # id to search for
    evals:list,                     # List of `Evaluation` objects to search
    results_path:str='data/results' # Path to save/load results
    ):                              # Report initialized from id
    "Create a Report by finding an evaluation matching the given id"
    ev = find_eval(evals, id, by='id')
    return cls(ev, pdf_url=eval_url(ev), results_path=results_path)

In [ ]:
#| export
@patch
def _repr_markdown_(self:Report) -> str:
    "Display report metadata and processing status in Jupyter notebooks"
    title = self.ev.meta.get('Title', 'Untitled')
    year = self.ev.meta.get('Year', 'n/a')
    org = self.ev.meta.get('Evaluation Commissioner', 'Unknown')
    
    pipeline = []
    if self.md_path: pipeline.append('✓ OCR')
    if self.curation_status == 'sections_selected': pipeline.append(f'✓ Curated ({len(self.selected_headings)} headings)')
    if self.mappings:
        mapped = ', '.join(self.mappings.keys())
        pipeline.append(f'✓ Mappings ({mapped})')
    pipeline_str = ' | '.join(pipeline) if pipeline else 'Not processed'
    
    return f"""
## Report: {title}
**Year:** {year} | **Organization:** {org}  
**ID:** `{self.id}`

**Pipeline:** {pipeline_str}  
**Curation:** {self.curation_status}

**Report:** [View Evaluation Report]({self.pdf_url})
"""

#### Creating reports

Create a report from a URL:

In [ ]:
#| eval: false
evals = load_evals('files/test/evaluations.json')

url = "https://evaluation.iom.int/sites/g/files/tmzbdl151/files/docs/resources/AAP%20Evaluation%20Report_final_.pdf"
report = Report.from_url(url, evals, results_path='files/test/results')
report


## Report: Evaluation of IOM Accountability to Affected Populations
**Year:** 2025 | **Organization:** IOM  
**ID:** `6c3c2cf3fa479112967612b0baddab72`

**Pipeline:** ✓ Curated (1 headings)  
**Curation:** sections_selected

**Report:** [View Evaluation Report](https://evaluation.iom.int/sites/g/files/tmzbdl151/files/docs/resources/AAP%20Evaluation%20Report_final_.pdf)


Or from a title:

In [ ]:
#| eval: false
title = 'Final Evaluation of the EU-IOM Joint Initiative for migrant protection and reintegration in the horn of Africa'
report = Report.from_title(title, evals, results_path='files/test/results')
report


## Report: Final Evaluation of the EU-IOM Joint Initiative for migrant protection and reintegration in the horn of Africa
**Year:** 2023 | **Organization:** IOM  
**ID:** `49d2fba781b6a7c0d94577479636ee6f`

**Pipeline:** ✓ Curated (5 headings) | ✓ Mappings (enbs, ccps, gcms, outs)  
**Curation:** sections_selected

**Report:** [View Evaluation Report](https://evaluation.iom.int/sites/g/files/tmzbdl151/files/docs/resources/Abridged%20Evaluation%20Report_%20Final_Olta%20NDOJA.pdf)


Or from an id:

In [ ]:
#| eval: false
id = '49d2fba781b6a7c0d94577479636ee6f'
report = Report.from_id(id, evals, results_path='files/test/results')
report


## Report: Final Evaluation of the EU-IOM Joint Initiative for migrant protection and reintegration in the horn of Africa
**Year:** 2023 | **Organization:** IOM  
**ID:** `49d2fba781b6a7c0d94577479636ee6f`

**Pipeline:** ✓ Curated (5 headings) | ✓ Mappings (enbs, ccps, gcms, outs)  
**Curation:** sections_selected

**Report:** [View Evaluation Report](https://evaluation.iom.int/sites/g/files/tmzbdl151/files/docs/resources/Abridged%20Evaluation%20Report_%20Final_Olta%20NDOJA.pdf)


::: {.callout-note}
When creating a report from title or id (rather than URL), `eval_url` is used to retrieve the main evaluation report from among the documents available for each evaluation (e.g. brief, annexes, management response).
:::

## Persistence

Reports automatically save after each pipeline stage. Use `load_report` to resume from any checkpoint.

In [ ]:
#| export
@patch
def save(self:Report,
         path:str=None  # Override default results path
        ) -> Report:    # Reports self for method chaining
    "Save report state to JSON"
    p = Path(path or self.results_path)/f'{self.id}.json'
    p.parent.mkdir(parents=True, exist_ok=True)
    data = dict(
        id=self.id,
        report_url=self.pdf_url,
        meta={**self.ev.meta, 'report_url': self.pdf_url},
        docs=self.ev.docs,
        curation_status=self.curation_status,
        selected_headings=self.selected_headings,
        mappings=self.mappings,
        timestamp=datetime.now().isoformat()
    )
    p.write_text(json.dumps(data, indent=2))
    return self

In [ ]:
#| export
def load_report(id:str,                  # Report ID (hash)
                base_path:str='data'     # Base directory containing pdf/, md/, results/
               ) -> Report:              # The loaded Report
    "Load a saved Report by id"
    results_path = Path(base_path)/'results'
    data = json.loads((results_path/f'{id}.json').read_text())
    
    ev = Evaluation(id=data['id'], meta=data['meta'], docs=data['docs'])
    report = Report(ev, pdf_url=data.get('report_url'), results_path=results_path)
    
    # Infer paths from convention
    pdf_dir = Path(base_path)/'pdf'/id
    md_dir = Path(base_path)/'md'/id
    report.pdf_path = pdf_dir if pdf_dir.exists() else None
    report.md_path = md_dir if md_dir.exists() else None
    
    # Restore new attributes
    report.curation_status = data.get('curation_status', 'pending')
    report.selected_headings = data.get('selected_headings', [])
    report.mappings = data.get('mappings', {})
    
    return report

#### Resuming from checkpoint

In [ ]:
#| eval: false
report = load_report('49d2fba781b6a7c0d94577479636ee6f', base_path='files/test')
report

## Pipeline Methods

The pipeline stages: `download` → `ocr` → `curate` → `map_*`

### Download

Downloads the evaluation PDF from IOM's repository.

In [ ]:
#| export
@patch
def download(self:Report,
             dst:str='data/pdf',  # Destination directory for PDFs
             force:bool=False      # Force re-download
            ) -> Report:           # Self for chaining
    "Download evaluation PDF to `dst`/`eval_id`/"
    if self.pdf_path and not force: return self
    self.pdf_path = download_eval(self.ev, dst=dst)
    self.save(self.results_path)
    return self

In [ ]:
#| eval: false
url = "https://evaluation.iom.int/sites/g/files/tmzbdl151/files/docs/resources/Abridged%20Evaluation%20Report_%20Final_Olta%20NDOJA.pdf"
report = Report.from_url(url, evals, results_path='files/test/results')
_ = report.download(dst='files/test/pdf')

In [ ]:
#| eval: false
Path('files/test/pdf/49d2fba781b6a7c0d94577479636ee6f').ls()

[Path('files/test/pdfs/49d2fba781b6a7c0d94577479636ee6f/Abridged%20Evaluation%20Report_%20Final_Olta%20NDOJA.pdf'), Path('files/test/pdfs/49d2fba781b6a7c0d94577479636ee6f/Final%20Evaluation%20Report%20Final_Olta%20NDOJA.pdf'), Path('files/test/pdfs/49d2fba781b6a7c0d94577479636ee6f/HoA%20EU%20JI%20Final%20Eval%20-%20Management%20Response%20Matrix%20-%20Final.pdf'), Path('files/test/pdfs/49d2fba781b6a7c0d94577479636ee6f/ISP_IOM_Case-Management-Return-Reintegr-JI-Review_final.pdf'), Path('files/test/pdfs/49d2fba781b6a7c0d94577479636ee6f/Evaluation%20Learning%20Brief_Final_Olta%20NDOJA.pdf')]

### OCR

Runs OCR on the PDF using Mistral's API and converts to markdown with proper heading hierarchy.

In [ ]:
#| export
@patch
async def ocr(self:Report,
              dst:str='data/md',       # Destination directory for markdown files
              add_img_desc:bool=True,  # Whether to add image descriptions
              force:bool=False,        # Force re-OCR              
              **kwargs                 # Additional args passed to pdf_to_md
             ) -> Report:              # Self for chaining
    "Run OCR on PDF and fix heading hierarchy"
    if self.md_path and not force: return self
    if self.pdf_path is None: raise ValueError("Call download() first")
    if self.pdf_url: pdf_file = self.pdf_path/Path(self.pdf_url).name
    else: pdf_file = first(self.pdf_path.glob('*.pdf'))
    await pdf_to_md(pdf_file, Path(dst)/self.id, add_img_desc=add_img_desc, **kwargs)
    self.md_path = Path(dst)/self.id
    self.save(self.results_path)
    return self

In [ ]:
#| eval: false
await report.ocr(dst='files/test/md', add_img_desc=False, force=False)


## Report: Final Evaluation of the EU-IOM Joint Initiative for migrant protection and reintegration in the horn of Africa
**Year:** 2023 | **Organization:** IOM  
**ID:** `49d2fba781b6a7c0d94577479636ee6f`

**Processing Status:**  
✓ PDF downloaded | ✓ Markdown converted | ✓ Sections extracted (~227 tokens) | ✓ Mappings: enablers, ccps, gcm, outputs, enbs, gcms, outs

**Report:** [View Evaluation Report](https://evaluation.iom.int/sites/g/files/tmzbdl151/files/docs/resources/Abridged%20Evaluation%20Report_%20Final_Olta%20NDOJA.pdf)


In [ ]:
#| eval: false
report.md_path

Path('files/test/md/49d2fba781b6a7c0d94577479636ee6f')

In [ ]:
#| eval: false
report.md_path.ls()[:2]

[Path('files/test/md/49d2fba781b6a7c0d94577479636ee6f/page_21.md'), Path('files/test/md/49d2fba781b6a7c0d94577479636ee6f/page_15.md')]

### Curate

Use the curator app (`06_curator.ipynb`) to review OCR'd headings and select relevant sections. Once complete, the report's `curation_status` will be `'sections_selected'` and `selected_headings` will contain the chosen headings.

In [ ]:
#| export
@patch
def get_sections(self:Report) -> str:
    "Extract sections on demand from selected headings"
    if self.md_path is None: raise ValueError("Call ocr() first")
    if self.curation_status != 'sections_selected': raise ValueError("Curation required: use curator app to select headings first")
    if not self.selected_headings: raise ValueError("No headings selected")
    return extract_sections(read_pgs(self.md_path), selected_headings=self.selected_headings)

#### Extracting sections

Once curation is complete, use `get_sections()` to extract the selected content:

In [ ]:
#| eval: false
report = load_report('49d2fba781b6a7c0d94577479636ee6f', base_path='files/test')
report.selected_headings

['## 1. Introduction ... page 4',
 '## 2. Background of the JI-HoA ... page 5',
 '## 3. Methodology ... page 8',
 '### 4.3. Effectiveness ... page 16',
 '## 5. Conclusions and Recommendations ... page 27']

In [ ]:
#| eval: false
print(report.get_sections()[:500])

# Final Evaluation of the EU-IOM Joint Initiative for migrant protection and reintegration in the horn of Africa ... page 1
## 1. Introduction ... page 4

In 2016, the EU and IOM launched the EU-IOM Joint Initiative for Migrant Protection and Reintegration, with as overall objective “To contribute to facilitating orderly, safe, regular and rights-based migration through the facilitation of dignified voluntary return and the implementation of development-focused and sustainable reintegration poli


## Thematic mapping

Map extracted sections to IOM's strategic frameworks (SRF and GCM). Each mapping method can be run independently after curation.

In [ ]:
#| export
@patch
def ensure_sys_blocks(self:Report) -> None:
    "Ensure system blocks are available"
    if not hasattr(self, '_sys_blocks'): self._sys_blocks = mk_system_blocks(self.get_sections())

In [ ]:
#| export
@delegates(map_themes)
def map_single(sys_blocks,                 # System blocks from mk_system_blocks
                theme_type,                # One of: 'enbs', 'ccps', 'gcms', 'outs'
                path=None,                 # Path to theme files
                model='claude-haiku-4-5',  # Model to use for mapping
                gcm_ids=None,              # GCM IDs for output mapping
                **kwargs                   # Additional args passed to map_themes
               ) -> dict:                  # Mapping results
    "Map system blocks (Report) to a single theme type using appropriate prompts and formatting"
    if theme_type == 'enbs': res = map_themes(sys_blocks, fmt_enbs(load_enbs(path)), load_prompt('srf_enablers'), model, **kwargs)
    elif theme_type == 'ccps': res = map_themes(sys_blocks, fmt_ccps(load_ccps(path)), load_prompt('srf_ccps'), model, **kwargs)
    elif theme_type == 'gcms': res = map_themes(sys_blocks, load_gcms(path), load_prompt('gcms'), model, **kwargs)
    elif theme_type == 'outs':
        srf_obj, gcm_lut = load_srf_outs(path), load_gcms_lut(path)
        output_ids = get_srf_outs(gcm_lut, gcm_ids)
        res = map_themes(sys_blocks, fmt_srf_outs(srf_obj, output_ids), load_prompt('srf_outputs'), model, **kwargs)
    return parse_res(res)

### Map enablers

Maps to Strategic Results Framework enablers (organizational capabilities).


In [ ]:
#| export
@patch
def map_enbs(self:Report,
             force:bool=False,  # Re-run even if already completed
             **kwargs           # Additional args passed to map_single (e.g. path, model)
            ) -> Report:        # Self for chaining
    "Map report sections to Strategic Results Framework enablers"
    if 'enbs' in self.mappings and not force: return self
    self.ensure_sys_blocks()
    self.mappings['enbs'] = map_single(self._sys_blocks, 'enbs', **kwargs)
    self.save(self.results_path)
    return self

Here let's consider we want to resume a curated report and run the mappings:

In [ ]:
#| eval: false
# Resuming where left
report = load_report('49d2fba781b6a7c0d94577479636ee6f', base_path='files/test')

# Mapping enablers
report.map_enbs(model='claude-haiku-4-5', force=False)


## Report: Final Evaluation of the EU-IOM Joint Initiative for migrant protection and reintegration in the horn of Africa
**Year:** 2023 | **Organization:** IOM  
**ID:** `49d2fba781b6a7c0d94577479636ee6f`

**Pipeline:** ✓ PDF | ✓ MD | ✓ Curated (5 headings) | ✓ Mappings (enbs)  
**Curation:** sections_selected

**Report:** [View Evaluation Report](https://evaluation.iom.int/sites/g/files/tmzbdl151/files/docs/resources/Abridged%20Evaluation%20Report_%20Final_Olta%20NDOJA.pdf)


In [ ]:
#| eval: false
sort_by_relevance(report.mappings['enbs'])[:2]

[{'theme_id': '4',
  'theme_title': 'Data and evidence',
  'relevance_score': 0.87,
  'reasoning': "This report would likely be essential for a synthesis on Enabler 4 (Data and Evidence). The evaluation explicitly examines IOM's data and evidence systems as a primary focus, with data availability and use appearing prominently in the program objectives, evaluation questions, and findings. Specific Objective 1 directly addresses 'Partner countries and relevant stakeholders developed or strengthened evidence-based return and reintegration procedures,' with substantial analysis of IOM's data production, data management systems, and evidence use. Key relevant sections include: the detailed findings on 'Data availability' (4.3.1.1) documenting that 'the JI exceeded the targets set for the number of field studies, surveys and other research conducted' and the 'increased availability of migration data...achieved mainly through the production and publication of migration data and research outpu

### Map CCPs

Maps to SRF Cross-Cutting Priorities.

In [ ]:
#| export
@patch
def map_ccps(self:Report,
             force:bool=False,  # Re-run even if already completed
             **kwargs           # Additional args passed to map_single (e.g. path, model)
            ) -> Report:        # Self for chaining
    "Map report sections to Strategic Results Framework cross-cutting priorities"
    if 'ccps' in self.mappings and not force: return self
    self.ensure_sys_blocks()
    self.mappings['ccps'] = map_single(self._sys_blocks, 'ccps', **kwargs)
    self.save(self.results_path)
    return self

In [ ]:
#| eval: false
report.map_ccps(model='claude-haiku-4-5', force=False)


## Report: Final Evaluation of the EU-IOM Joint Initiative for migrant protection and reintegration in the horn of Africa
**Year:** 2023 | **Organization:** IOM  
**ID:** `49d2fba781b6a7c0d94577479636ee6f`

**Pipeline:** ✓ PDF | ✓ MD | ✓ Curated (5 headings) | ✓ Mappings (enbs, ccps)  
**Curation:** sections_selected

**Report:** [View Evaluation Report](https://evaluation.iom.int/sites/g/files/tmzbdl151/files/docs/resources/Abridged%20Evaluation%20Report_%20Final_Olta%20NDOJA.pdf)


In [ ]:
#| eval: false
sort_by_relevance(report.mappings['ccps'])[:2]

[{'theme_id': '3',
  'theme_title': 'Protection-centred',
  'relevance_score': 0.79,
  'reasoning': "This report would likely contribute meaningful evidence to a synthesis on Cross-cutting Priority 3 (Protection-centred). The evaluation explicitly examines protection-related outcomes and approaches as core components of the program. The program's overarching objective directly references 'migrant protection' and 'rights-based migration,' and the evaluation assesses 'protection and voluntary assisted return' as one of five pillars of action. Multiple findings substantively address protection-centred commitments: Section 4.3.2 ('Safe, humane, dignified voluntary return processes') evaluates IOM's effectiveness in providing protection to stranded migrants, with findings showing 95% of assisted migrants satisfied with travel arrangements and 99.6% reporting safe, well-organized travel. The report documents extensive evidence of 'life-saving benefits for beneficiaries' and notes that return

### Map GCM objectives

Maps to Global Compact for Migration objectives.

In [ ]:
#| export
@patch
def map_gcms(self:Report,
             force:bool=False,  # Re-run even if already completed
             **kwargs           # Additional args passed to map_single (e.g. path, model)
            ) -> Report:        # Self for chaining
    "Map report sections to Global Compact for Migration objectives"
    if 'gcms' in self.mappings and not force: return self
    self.ensure_sys_blocks()
    self.mappings['gcms'] = map_single(self._sys_blocks, 'gcms', **kwargs)
    self.save(self.results_path)
    return self

In [ ]:
#| eval: false
report.map_gcms(model='claude-haiku-4-5', force=False)


## Report: Final Evaluation of the EU-IOM Joint Initiative for migrant protection and reintegration in the horn of Africa
**Year:** 2023 | **Organization:** IOM  
**ID:** `49d2fba781b6a7c0d94577479636ee6f`

**Pipeline:** ✓ PDF | ✓ MD | ✓ Curated (5 headings) | ✓ Mappings (enbs, ccps, gcms)  
**Curation:** sections_selected

**Report:** [View Evaluation Report](https://evaluation.iom.int/sites/g/files/tmzbdl151/files/docs/resources/Abridged%20Evaluation%20Report_%20Final_Olta%20NDOJA.pdf)


In [ ]:
#| eval: false
sort_by_relevance(report.mappings['gcms'])[:2]



[{'theme_id': '21',
  'theme_title': 'Cooperate In Facilitating Safe And Dignified Return And Readmission, As Well As Sustainable Reintegration',
  'relevance_score': 0.92,
  'reasoning': "FUNDAMENTAL. This report would be essential evidence for a synthesis on GCM Objective 21. Safe, dignified return and sustainable reintegration are the core themes of the entire evaluation. The programme's overarching objective is 'To contribute to facilitating orderly, safe, regular and rights-based migration through the facilitation of dignified voluntary return and the implementation of development-focused and sustainable reintegration policies and processes' (page 4). The evaluation explicitly examines Specific Objective 2 ('Safe, humane, dignified voluntary return processes are enhanced along main migration routes,' pages 18-20) and Specific Objective 3 ('Returnees are sustainably integrated in host communities,' pages 20-23). Key findings show that the JI supported 9,025 migrants to return volun

### Map outputs

Maps to SRF outputs. If `gcm_ids` not provided, uses the top GCM objective from prior mapping.

In [ ]:
#| export
@patch
def map_outs(self:Report,
             gcm_ids=None,      # GCM IDs to filter SRF objectives
             force:bool=False,  # Re-run even if already completed
             **kwargs           # Additional args passed to map_single (e.g. path, model)
            ) -> Report:        # Self for chaining
    "Map report sections to Strategic Results Framework outputs"
    if 'outs' in self.mappings and not force: return self
    self.ensure_sys_blocks()
    if gcm_ids is None:
        top_ids = get_top_ids(self.mappings.get('gcms', []))
        if not top_ids: return self
        gcm_ids = [top_ids[0]]
    self.mappings['outs'] = map_single(self._sys_blocks, 'outs', gcm_ids=gcm_ids, **kwargs)
    self.save(self.results_path)
    return self

In [ ]:
#| eval: false
report.map_outs(model='claude-haiku-4-5', force=False)


## Report: Final Evaluation of the EU-IOM Joint Initiative for migrant protection and reintegration in the horn of Africa
**Year:** 2023 | **Organization:** IOM  
**ID:** `49d2fba781b6a7c0d94577479636ee6f`

**Pipeline:** ✓ PDF | ✓ MD | ✓ Curated (5 headings) | ✓ Mappings (enbs, ccps, gcms, outs)  
**Curation:** sections_selected

**Report:** [View Evaluation Report](https://evaluation.iom.int/sites/g/files/tmzbdl151/files/docs/resources/Abridged%20Evaluation%20Report_%20Final_Olta%20NDOJA.pdf)


In [ ]:
#| eval: false
sort_by_relevance(report.mappings['outs'])[:2]

[{'theme_id': '2b63',
  'theme_title': 'Returning migrants and returning, relocated and locally integrating displaced persons receive reintegration assistance in line with their needs and those of broader community members.',
  'relevance_score': 0.89,
  'reasoning': "This report would likely be essential for a synthesis on Output 2b63 (reintegration assistance for returning migrants and displaced persons). The evaluation explicitly examines IOM's reintegration programming across the Horn of Africa, with reintegration assistance appearing prominently in the program title, evaluation objectives, and throughout the report's structure. The report dedicates substantial analysis to individual and community-based reintegration across three dimensions (economic, social, psychosocial), directly addressing this Output's core deliverables. Key findings in Section 4.3.3 provide extensive evidence on reintegration assistance delivery: 15,161 beneficiaries received reintegration support (exceeding 

### Map all themes

Convenience method to run all mapping stages in sequence.

In [ ]:
#| export
@patch
def map_all(self:Report,
            **kwargs  # Args passed to all mapping methods
           ) -> Report:  # Self for chaining
    "Run all theme mappings in sequence"
    return self.map_enbs(**kwargs).map_ccps(**kwargs).map_gcms(**kwargs).map_outs(**kwargs)

## Run full pipeline

Run the pipeline on a single evaluation report. The pipeline runs until curation is needed, then continues after curation is complete. Re-run the same call after using the curator app to complete all stages.

In [ ]:
#| export
def should_force(force,     # Bool to force all steps, or set of step names to force
                 step       # Step name to check
                ) -> bool:  # Whether to force the step
    "Check if step should be forced - handles bool or set of step names"
    if isinstance(force, bool): return force
    return step in force

In [ ]:
#| export
class PipelineResult:
    def __init__(self, report, status, error=None, step=None):
        store_attr()

In [ ]:
#| export
def _get_current_step(report):
    "Infer which step the report is at based on its state"
    if not report.md_path: return 'ocr'
    if report.curation_status != 'sections_selected': return 'curation'
    return 'mapping'

In [ ]:
#| export
async def run_pipeline(evals:list,                  # List of `Evaluation` objects to search
                       url:str=None,                # URL of the evaluation PDF
                       id:str=None,                 # Evaluation ID
                       title:str=None,              # Evaluation title
                       base_path:str='data',        # Base directory (contains pdf/, md/, results/)
                       ocr_kwargs:dict=None,        # Additional arguments passed to ocr
                       force:bool|set=False,        # Force re-run: True for all, or set of step names
                       delete_pdf:bool=True,        # Delete PDF after OCR
                       **kwargs                     # Additional arguments passed to mapping functions
                      ) -> PipelineResult:          # Pipeline result with status and report
    "Run pipeline as far as possible: download → ocr → [curate] → map_all"
    if sum(x is not None for x in (url, id, title)) != 1:
        raise ValueError("Provide exactly one of: url, id, title")
    base = Path(base_path)
    pdf_dst, md_dst, results_path = base/'pdf', base/'md', base/'results'
    
    if url: report = Report.from_url(url, evals, results_path=results_path)
    elif id: report = Report.from_id(id, evals, results_path=results_path)
    else: report = Report.from_title(title, evals, results_path=results_path)
    
    try: report = load_report(report.id, base_path=base_path)
    except FileNotFoundError: pass
    
    try:
        if not report.md_path or should_force(force, 'ocr'):
            if not report.pdf_path or should_force(force, 'download'):
                logger.info(f"Downloading PDF...")
                report.download(dst=pdf_dst, force=should_force(force, 'download'))
            
            logger.info(f"Running OCR...")
            await report.ocr(dst=md_dst, force=should_force(force, 'ocr'), **(ocr_kwargs or {}))


        if delete_pdf and report.pdf_path:
            shutil.rmtree(report.pdf_path)
            report.pdf_path = None
                
        if report.curation_status != 'sections_selected':
            logger.info(f"Awaiting curation. Use curator app to select headings, then re-run.")
            return PipelineResult(report, 'awaiting_curation', step='curation')
        
        logger.info(f"Mapping enablers...")
        report.map_enbs(force=should_force(force, 'enbs'), **kwargs)
        logger.info(f"Mapping CCPs...")
        report.map_ccps(force=should_force(force, 'ccps'), **kwargs)
        logger.info(f"Mapping GCM objectives...")
        report.map_gcms(force=should_force(force, 'gcms'), **kwargs)
        logger.info(f"Mapping outputs...")
        report.map_outs(force=should_force(force, 'outs'), **kwargs)
        logger.info(f"Pipeline complete!")
        return PipelineResult(report, 'completed')
    
    except Exception as e:
        step = _get_current_step(report)
        return PipelineResult(report, 'failed', error=str(e), step=step)


In [ ]:
#| eval: false
evals = load_evals('files/test/evaluations.json')
id = '49d2fba781b6a7c0d94577479636ee6f'
report = await run_pipeline(evals, id=id, base_path='files/test', ocr_kwargs=dict(add_img_desc=False), 
                            force=False, model='claude-haiku-4-5')
report

__main__ - INFO - Mapping enablers...


__main__ - INFO - Mapping CCPs...


__main__ - INFO - Mapping GCM objectives...


__main__ - INFO - Mapping outputs...


__main__ - INFO - Pipeline complete!



## Report: Final Evaluation of the EU-IOM Joint Initiative for migrant protection and reintegration in the horn of Africa
**Year:** 2023 | **Organization:** IOM  
**ID:** `49d2fba781b6a7c0d94577479636ee6f`

**Pipeline:** ✓ PDF | ✓ MD | ✓ Curated (5 headings) | ✓ Mappings (enbs, ccps, gcms, outs)  
**Curation:** sections_selected

**Report:** [View Evaluation Report](https://evaluation.iom.int/sites/g/files/tmzbdl151/files/docs/resources/Abridged%20Evaluation%20Report_%20Final_Olta%20NDOJA.pdf)


## Batch

In [ ]:
#| export
def get_report_urls():
    data = httpx.get('https://evaluation.iom.int/json/api/evaluation').json()
    return {f"https://evaluation.iom.int{e['download_url']}" for e in data['Evaluations']}

In [ ]:
#| export
report_urls = get_report_urls()

In [ ]:
#| export
def get_eval_report_url(r, report_urls):
    for d in r.docs:
        if d['url'] in report_urls:
            return d['url']
    return None

In [ ]:
#| export
class BatchResult:
    def __init__(self, completed=None, awaiting_curation=None, failed=None, skipped=None):
        store_attr()
        self.completed = completed or []
        self.awaiting_curation = awaiting_curation or []
        self.failed = failed or []
        self.skipped = skipped or []
    
    def save(self, path):
        Path(path).parent.mkdir(parents=True, exist_ok=True)
        Path(path).write_text(json.dumps(self.__dict__, indent=2))

In [ ]:
#| export
def _filter_evals(evals, year=None, ids=None):
    if ids: return [e for e in evals if e.id in ids]
    if year: return [e for e in evals if e.meta.get('Year') == year]
    return evals


In [ ]:
#| export
def _is_completed(report):
    required = {'enbs', 'ccps', 'gcms', 'outs'}
    return required.issubset(report.mappings.keys()) if report.mappings else False

In [ ]:
#| export
async def batch_run(evals, report_urls, base_path='../../data', year=None, ids=None, 
                    force=None, stop_after=None, delete_pdf=False) -> BatchResult:
    "Run pipeline on a batch of reports"
    filtered = _filter_evals(evals, year=year, ids=ids)
    result = BatchResult()
    
    for ev in filtered:
        try:
            report = load_report(ev.id, base_path=base_path)
        except FileNotFoundError:
            report = None
        
        # Skip if already completed (unless forcing)
        if report and _is_completed(report) and not force:
            result.skipped.append(ev.id)
            continue
        
        # Get report URL
        url = get_eval_report_url(ev, report_urls)
        if not url:
            result.failed.append({'id': ev.id, 'step': 'url_lookup', 'error': 'Report URL not found'})
            continue
        
        # Run pipeline
        pr = await run_pipeline(evals, url=url, base_path=base_path, 
                                force=force, delete_pdf=delete_pdf)
        
        if pr.status == 'completed': result.completed.append(ev.id)
        elif pr.status == 'awaiting_curation': result.awaiting_curation.append(ev.id)
        elif pr.status == 'failed': result.failed.append({'id': ev.id, 'step': pr.step, 'error': pr.error})

    timestamp = datetime.now().strftime('%Y-%m-%d_%H-%M')
    save_path = Path(base_path)/'batch_runs'/f'{timestamp}.json'
    result.save(save_path)
        
    return result


In [ ]:
test_ids = ['4341695461234eee3deb51ac68871109',
            '6c3c2cf3fa479112967612b0baddab72',
            '49d2fba781b6a7c0d94577479636ee6f']

In [ ]:
!ls ../data

md  pdf  results  results-bu  results_test


In [ ]:
result = await batch_run(evals, report_urls, base_path='../../data', ids=test_ids)
result